# Hearthline ARC-AGI-2 format baseline

Prepared for Christopher D. Pang. This cleared notebook is a deterministic, standard-library-only format check. It is not a competitive solver, run authorization, submission, or score claim.

In [ ]:
import json
import os
import re
import time
from pathlib import Path

COMPETITION = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2')
CHALLENGES_PATH = COMPETITION / 'arc-agi_test_challenges.json'
SAMPLE_PATH = COMPETITION / 'sample_submission.json'
OUTPUT_PATH = Path('/kaggle/working/submission.json')
DEADLINE = time.monotonic() + 11 * 60 * 60


In [ ]:
def valid_grid(value):
    if not isinstance(value, list) or not 1 <= len(value) <= 30:
        return False
    if not all(isinstance(row, list) for row in value):
        return False
    widths = {len(row) for row in value}
    if len(widths) != 1 or not 1 <= next(iter(widths)) <= 30:
        return False
    return all(
        isinstance(cell, int) and not isinstance(cell, bool) and 0 <= cell <= 9
        for row in value for cell in row
    )

def validate_challenges(value):
    if not isinstance(value, dict) or not value:
        raise ValueError('challenge set must be a nonempty object')
    for task_id, task in value.items():
        if not isinstance(task_id, str) or re.fullmatch(r'[0-9a-f]{8}', task_id) is None:
            raise ValueError('invalid task id')
        if not isinstance(task, dict) or set(task) != {'train', 'test'}:
            raise ValueError('invalid task object')
        if not isinstance(task['train'], list) or not task['train']:
            raise ValueError('training demonstrations required')
        if not isinstance(task['test'], list) or not task['test']:
            raise ValueError('test inputs required')
        for pair in task['train']:
            if not isinstance(pair, dict) or set(pair) != {'input', 'output'}:
                raise ValueError('invalid demonstration')
            if not valid_grid(pair['input']) or not valid_grid(pair['output']):
                raise ValueError('invalid demonstration grid')
        for pair in task['test']:
            if not isinstance(pair, dict) or set(pair) != {'input'}:
                raise ValueError('solver view must contain test input only')
            if not valid_grid(pair['input']):
                raise ValueError('invalid test grid')

def baseline_pair(input_grid):
    copied = [row[:] for row in input_grid]
    zero = [[0 for _ in row] for row in input_grid]
    return {'attempt_1': copied, 'attempt_2': zero}

def validate_submission(challenges, submission):
    if not isinstance(submission, dict) or set(submission) != set(challenges):
        raise ValueError('submission task coverage mismatch')
    for task_id, task in challenges.items():
        records = submission[task_id]
        if not isinstance(records, list) or len(records) != len(task['test']):
            raise ValueError('submission test-input coverage mismatch')
        for record in records:
            if not isinstance(record, dict) or set(record) != {'attempt_1', 'attempt_2'}:
                raise ValueError('exactly two attempts required')
            if not valid_grid(record['attempt_1']) or not valid_grid(record['attempt_2']):
                raise ValueError('invalid attempt grid')


In [ ]:
challenges = json.loads(CHALLENGES_PATH.read_text(encoding='utf-8'))
validate_challenges(challenges)
submission = {}
for task_id in sorted(challenges):
    # The format baseline is also the declared label-free timeout fallback.
    if time.monotonic() >= DEADLINE:
        submission[task_id] = [baseline_pair(pair['input']) for pair in challenges[task_id]['test']]
        continue
    submission[task_id] = [baseline_pair(pair['input']) for pair in challenges[task_id]['test']]
validate_submission(challenges, submission)
if not SAMPLE_PATH.is_file():
    raise FileNotFoundError('required sample_submission.json is missing')
sample = json.loads(SAMPLE_PATH.read_text(encoding='utf-8'))
validate_submission(challenges, sample)
payload = json.dumps(submission, sort_keys=True, separators=(',', ':')) + '\n'
temporary = OUTPUT_PATH.with_suffix('.json.tmp')
with temporary.open('w', encoding='utf-8', newline='\n') as stream:
    stream.write(payload)
    stream.flush()
    os.fsync(stream.fileno())
temporary.replace(OUTPUT_PATH)
